# Starts-at-Zero Checker — Clean Tesseract OCR Version

This notebook uses **Tesseract OCR only**.  
EasyOCR has been removed to avoid mixed coordinate systems and duplicated OCR logic.

Main idea:

1. Crop only the likely value-axis area.
2. Run Tesseract with several preprocessing variants.
3. Convert every OCR box back to the **original image coordinate system**.
4. Group detected numbers into a value axis.
5. Decide whether the axis contains `0`.

For vertical bar charts, the value axis is usually the **left y-axis**.  
For horizontal bar charts, the value axis is usually the **bottom x-axis**.

In [ ]:
# Install Python packages if needed.
# Important: pytesseract is only the Python wrapper.
# You must also install the real Tesseract engine on your computer.
#
# macOS:
#   brew install tesseract
#
# Windows:
#   Install Tesseract from UB Mannheim, then set TESSERACT_CMD below.
#
# Linux:
#   sudo apt install tesseract-ocr

%pip install opencv-python numpy pillow pytesseract pandas matplotlib

In [1]:
import os
import re
from pathlib import Path
from shutil import which
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import pytesseract
from pytesseract import Output
from PIL import Image as PILImage
import matplotlib.pyplot as plt

## 1. Tesseract configuration

In [2]:
# Change this only if Tesseract is installed somewhere else.
# On Apple Silicon Macs, /opt/homebrew/bin/tesseract is common.
# On Intel Macs, /usr/local/bin/tesseract is common.

TESSERACT_CMD = (
    os.getenv("TESSERACT_CMD")
    or which("tesseract")
    or "/opt/homebrew/bin/tesseract"
)

pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD

def assert_tesseract_available():
    cmd = pytesseract.pytesseract.tesseract_cmd
    if cmd is None:
        raise RuntimeError("Tesseract command path is not set.")

    if not Path(cmd).exists() and which("tesseract") is None:
        raise RuntimeError(
            "Tesseract engine was not found. "
            "Install it with `brew install tesseract` on macOS, "
            "or set TESSERACT_CMD to the correct executable path."
        )

    try:
        version = pytesseract.get_tesseract_version()
        print(f"Tesseract found: {version}")
        print(f"Tesseract command: {cmd}")
    except Exception as exc:
        raise RuntimeError(
            f"Tesseract exists but could not be executed: {cmd}\n{exc}"
        )

assert_tesseract_available()

Tesseract found: 5.5.2
Tesseract command: /opt/homebrew/bin/tesseract


## 2. OCR text cleaning and numeric parsing

In [3]:
ZERO_VALUE_TOLERANCE = 0.5  # A detected value with abs(value) < 0.5 is treated as zero.

def normalize_ocr_text(text):
    """
    Normalize common OCR mistakes before numeric extraction.
    Keep this conservative. Do not convert too many letters into digits,
    otherwise false positives become more likely.
    """
    if text is None:
        return ""

    text = str(text).strip()

    # Normalize minus symbols and common OCR mistakes for zero.
    text = (
        text.replace("−", "-")
            .replace("–", "-")
            .replace("—", "-")
            .replace("O", "0")
            .replace("o", "0")
    )

    # Remove symbols that are common in chart labels.
    text = (
        text.replace(",", "")
            .replace("%", "")
            .replace("m", "")
            .replace("M", "")
            .strip()
    )

    return text


def extract_number(text):
    """
    Extract the first numeric value from OCR text.
    Examples:
        '0%'    -> 0.0
        '100%'  -> 100.0
        '1,000' -> 1000.0
        'O%'    -> 0.0
    """
    cleaned = normalize_ocr_text(text)

    # Require at least one digit after normalization.
    if not re.search(r"\d", cleaned):
        return None

    match = re.search(r"-?\d+(?:\.\d+)?", cleaned)
    if not match:
        return None

    try:
        return float(match.group(0))
    except ValueError:
        return None


def is_zero_value(value):
    return value is not None and abs(value) < ZERO_VALUE_TOLERANCE


def unique_sorted_values(values, tolerance=0.5):
    """
    Remove near-duplicate numeric values, then sort.
    Useful when Tesseract sees the same label in multiple preprocessing passes.
    """
    cleaned = sorted(float(v) for v in values)
    unique = []
    for v in cleaned:
        if not unique or abs(v - unique[-1]) > tolerance:
            unique.append(v)
    return unique

## 3. Image preprocessing

The important part is that every resized OCR result is converted back to original image coordinates later.

In [4]:
def read_image(image_path):
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Image not found or unreadable: {image_path}")
    return image


def to_gray(image):
    if len(image.shape) == 2:
        return image
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)


def preprocess_variants(roi_bgr, scale=3):
    """
    Return multiple OCR-ready versions of one ROI.

    Each output image has the same scale factor. Bounding boxes from Tesseract
    must be divided by this scale to return to the original ROI coordinates.
    """
    gray = to_gray(roi_bgr)

    resized = cv2.resize(
        gray,
        None,
        fx=scale,
        fy=scale,
        interpolation=cv2.INTER_CUBIC
    )

    variants = []

    # 1. Raw enlarged grayscale.
    variants.append(("gray", resized))

    # 2. CLAHE improves low-contrast axis labels.
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    clahe_img = clahe.apply(resized)
    variants.append(("clahe", clahe_img))

    # 3. Slight blur after CLAHE can reduce noisy pixels.
    blur = cv2.GaussianBlur(clahe_img, (3, 3), 0)
    variants.append(("clahe_blur", blur))

    # 4. Otsu threshold.
    # Tesseract usually prefers dark text on a light background.
    thresh_type = cv2.THRESH_BINARY + cv2.THRESH_OTSU
    if np.mean(blur) < 128:
        thresh_type = cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU

    _, otsu = cv2.threshold(blur, 0, 255, thresh_type)
    variants.append(("otsu", otsu))

    # 5. Adaptive threshold can help with uneven lighting/backgrounds.
    adaptive = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31,
        8
    )
    variants.append(("adaptive", adaptive))

    return variants


def show_image(path_or_bgr, title=None, figsize=(10, 6)):
    """
    Small helper for notebook display.
    """
    if isinstance(path_or_bgr, (str, Path)):
        image = cv2.imread(str(path_or_bgr))
    else:
        image = path_or_bgr

    if image is None:
        raise ValueError("Could not display image.")

    if len(image.shape) == 2:
        display_img = image
        cmap = "gray"
    else:
        display_img = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        cmap = None

    plt.figure(figsize=figsize)
    plt.imshow(display_img, cmap=cmap)
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()

## 4. Region-of-interest selection

This avoids most false positives from data labels inside the bars.

In [5]:
def make_axis_rois(image, orientation):
    """
    Return likely value-axis regions.

    orientation='vertical':
        vertical bars -> numeric value axis is usually on the left y-axis.

    orientation='horizontal':
        horizontal bars -> numeric value axis is usually at the bottom x-axis.
    """
    h, w = image.shape[:2]

    if orientation == "vertical":
        return [
            {
                "name": "left_axis_wide",
                "x0": 0,
                "y0": 0,
                "x1": int(w * 0.35),
                "y1": h,
                "scale": 3,
                "psm_modes": [4, 6, 11],
            },
            {
                "name": "left_axis_narrow",
                "x0": 0,
                "y0": 0,
                "x1": int(w * 0.22),
                "y1": h,
                "scale": 4,
                "psm_modes": [4, 6],
            },
            {
                "name": "bottom_left_zero_rescue",
                "x0": 0,
                "y0": int(h * 0.55),
                "x1": int(w * 0.40),
                "y1": h,
                "scale": 4,
                "psm_modes": [6, 11, 13],
            },
        ]

    if orientation == "horizontal":
        return [
            {
                "name": "bottom_axis_wide",
                "x0": 0,
                "y0": int(h * 0.55),
                "x1": w,
                "y1": h,
                "scale": 3,
                "psm_modes": [6, 11],
            },
            {
                "name": "bottom_left_zero_rescue",
                "x0": 0,
                "y0": int(h * 0.55),
                "x1": int(w * 0.45),
                "y1": h,
                "scale": 4,
                "psm_modes": [6, 11, 13],
            },
        ]

    raise ValueError("orientation must be either 'vertical' or 'horizontal'")

## 5. Tesseract OCR on chart-axis regions

In [6]:
TESSERACT_WHITELIST = "0123456789.,%mMOo-"

def tesseract_config(psm):
    return (
        f"--oem 3 --psm {psm} "
        f"-c tessedit_char_whitelist={TESSERACT_WHITELIST} "
        "-c classify_bln_numeric_mode=1 "
        "-c preserve_interword_spaces=1"
    )


def parse_confidence(value):
    try:
        return float(value)
    except Exception:
        return -1.0


def run_tesseract_on_processed_image(processed_img, psm, min_conf=-5):
    """
    Return raw Tesseract word-level detections for one processed ROI image.
    Coordinates are still in processed-image coordinates.
    """
    pil_img = PILImage.fromarray(processed_img)

    data = pytesseract.image_to_data(
        pil_img,
        config=tesseract_config(psm),
        output_type=Output.DICT
    )

    detections = []
    for i, raw_text in enumerate(data["text"]):
        text = str(raw_text).strip()
        if not text:
            continue

        value = extract_number(text)
        if value is None:
            continue

        conf = parse_confidence(data["conf"][i])
        if conf < min_conf:
            continue

        x = int(data["left"][i])
        y = int(data["top"][i])
        bw = int(data["width"][i])
        bh = int(data["height"][i])

        # Ignore unusably tiny detections.
        if bw <= 1 or bh <= 1:
            continue

        detections.append({
            "text": text,
            "value": value,
            "confidence": conf / 100.0 if conf > 1 else conf,
            "left_processed": x,
            "top_processed": y,
            "width_processed": bw,
            "height_processed": bh,
            "psm": psm,
        })

    return detections


def detections_are_duplicates(a, b, pixel_tolerance=10, value_tolerance=0.5):
    same_value = abs(a["value"] - b["value"]) <= value_tolerance
    close_position = (
        abs(a["cx"] - b["cx"]) <= pixel_tolerance
        and abs(a["cy"] - b["cy"]) <= pixel_tolerance
    )
    return same_value and close_position


def dedupe_numeric_detections(detections):
    """
    Keep the strongest version when the same label is detected by multiple
    preprocessing variants or PSM modes.
    """
    if not detections:
        return []

    # Prefer higher confidence, then larger boxes.
    sorted_dets = sorted(
        detections,
        key=lambda d: (d["confidence"], d["width"] * d["height"]),
        reverse=True
    )

    kept = []
    for det in sorted_dets:
        duplicate = any(detections_are_duplicates(det, existing) for existing in kept)
        if not duplicate:
            kept.append(det)

    # Return in reading-ish order.
    kept = sorted(kept, key=lambda d: (d["cy"], d["cx"]))
    return kept


def ocr_numbers_with_tesseract(image, orientation):
    """
    OCR all likely value-axis ROIs and return numeric boxes in ORIGINAL image coordinates.
    """
    rois = make_axis_rois(image, orientation)
    all_detections = []

    for roi in rois:
        x0, y0, x1, y1 = roi["x0"], roi["y0"], roi["x1"], roi["y1"]
        scale = roi["scale"]

        crop = image[y0:y1, x0:x1]
        if crop.size == 0:
            continue

        for preprocess_name, processed in preprocess_variants(crop, scale=scale):
            for psm in roi["psm_modes"]:
                raw_detections = run_tesseract_on_processed_image(processed, psm=psm)

                for det in raw_detections:
                    left = x0 + det["left_processed"] / scale
                    top = y0 + det["top_processed"] / scale
                    width = det["width_processed"] / scale
                    height = det["height_processed"] / scale
                    right = left + width
                    bottom = top + height

                    all_detections.append({
                        "value": det["value"],
                        "text": det["text"],
                        "confidence": float(det["confidence"]),
                        "left": float(left),
                        "right": float(right),
                        "top": float(top),
                        "bottom": float(bottom),
                        "cx": float(left + width / 2),
                        "cy": float(top + height / 2),
                        "width": float(width),
                        "height": float(height),
                        "roi": roi["name"],
                        "preprocess": preprocess_name,
                        "psm": psm,
                    })

    return dedupe_numeric_detections(all_detections)

## 6. Axis-label grouping logic

In [7]:
def is_monotonic(values):
    if len(values) < 2:
        return False

    increasing = all(values[i] <= values[i + 1] for i in range(len(values) - 1))
    decreasing = all(values[i] >= values[i + 1] for i in range(len(values) - 1))
    return increasing or decreasing


def looks_like_tick_sequence(values, tolerance_ratio=0.30):
    """
    Check whether numeric axis labels look like a regular tick sequence.

    Allows missing OCR ticks.
    Example:
        [0, 20, 40, 80, 100] can still pass because 40->80 is a missing 60.
    """
    vals = unique_sorted_values(values)

    if len(vals) < 3:
        return False

    diffs = [vals[i + 1] - vals[i] for i in range(len(vals) - 1)]
    diffs = [d for d in diffs if d > 0]

    if len(diffs) < 2:
        return False

    base_step = min(diffs)
    if base_step <= 0:
        return False

    for diff in diffs:
        multiple = max(1, round(diff / base_step))
        expected = multiple * base_step
        allowed_error = max(1.0, abs(expected) * tolerance_ratio)
        if abs(diff - expected) > allowed_error:
            return False

    return True


def collapse_same_y_labels(group, y_tolerance=10):
    """
    For vertical axes, multiple OCR variants can survive deduplication.
    Keep one label per y-position/value.
    """
    if not group:
        return []

    ordered = sorted(
        group,
        key=lambda d: (d["cy"], -d["confidence"])
    )

    collapsed = []
    for item in ordered:
        duplicate = False
        for kept in collapsed:
            same_y = abs(item["cy"] - kept["cy"]) <= y_tolerance
            same_value = abs(item["value"] - kept["value"]) <= 0.5
            if same_y and same_value:
                duplicate = True
                break
        if not duplicate:
            collapsed.append(item)

    return collapsed


def find_vertical_value_axis(numbers, image_width, image_height):
    """
    Find the left y-axis label group for vertical bar charts.
    """
    if len(numbers) < 3:
        return []

    groups = defaultdict(list)

    # Y-axis labels are usually right-aligned, so group by right edge.
    bin_size = max(12, image_width * 0.035)

    for item in numbers:
        if item["cx"] > image_width * 0.45:
            continue
        x_bin = int(item["right"] // bin_size)
        groups[x_bin].append(item)

    best_group = []
    best_score = -1

    for group in groups.values():
        group = collapse_same_y_labels(group)
        if len(group) < 3:
            continue

        group = sorted(group, key=lambda d: d["cy"])
        values_top_to_bottom = [g["value"] for g in group]

        x_positions = [g["right"] for g in group]
        y_positions = [g["cy"] for g in group]

        mean_x = float(np.mean(x_positions))
        x_spread = max(x_positions) - min(x_positions)
        y_spread = max(y_positions) - min(y_positions)

        score = 0

        # Must be on the left side.
        if mean_x < image_width * 0.35:
            score += 3

        # Axis labels should have similar right edge.
        if x_spread < image_width * 0.16:
            score += 3

        # Axis labels should be vertically spread.
        if y_spread > image_height * 0.25:
            score += 3

        # More ticks are better.
        score += min(len(group), 6)

        # Axis values should be monotonic in screen order.
        if is_monotonic(values_top_to_bottom):
            score += 4

        # Numeric values should look like a regular tick sequence.
        if looks_like_tick_sequence(values_top_to_bottom):
            score += 4

        # Bottom-most axis label is normally in the lower half.
        if max(y_positions) > image_height * 0.55:
            score += 2

        if score > best_score:
            best_score = score
            best_group = group

    # Do not accept weak/random groups.
    if best_score < 13:
        return []

    return best_group


def find_horizontal_value_axis(numbers, image_width, image_height):
    """
    Find the bottom x-axis value-label group for horizontal bar charts.
    """
    if len(numbers) < 3:
        return []

    groups = defaultdict(list)

    # X-axis labels have similar vertical position, so group by y center.
    bin_size = max(12, image_height * 0.05)

    for item in numbers:
        if item["cy"] < image_height * 0.45:
            continue
        y_bin = int(item["cy"] // bin_size)
        groups[y_bin].append(item)

    best_group = []
    best_score = -1

    for group in groups.values():
        if len(group) < 3:
            continue

        group = sorted(group, key=lambda d: d["cx"])
        values_left_to_right = [g["value"] for g in group]

        x_positions = [g["cx"] for g in group]
        y_positions = [g["cy"] for g in group]

        x_spread = max(x_positions) - min(x_positions)
        y_spread = max(y_positions) - min(y_positions)
        mean_y = float(np.mean(y_positions))

        score = 0

        # Must be near the bottom half.
        if mean_y > image_height * 0.55:
            score += 3

        # X-axis tick labels should be horizontally spread.
        if x_spread > image_width * 0.25:
            score += 3

        # Tick labels should be close to the same baseline.
        if y_spread < image_height * 0.12:
            score += 3

        score += min(len(group), 6)

        if is_monotonic(values_left_to_right):
            score += 4

        if looks_like_tick_sequence(values_left_to_right):
            score += 4

        if score > best_score:
            best_score = score
            best_group = group

    if best_score < 13:
        return []

    return best_group


def find_value_axis(numbers, orientation, image_width, image_height):
    if orientation == "vertical":
        return find_vertical_value_axis(numbers, image_width, image_height)

    if orientation == "horizontal":
        return find_horizontal_value_axis(numbers, image_width, image_height)

    raise ValueError("orientation must be either 'vertical' or 'horizontal'")

## 7. Zero detection and final compliance check

In [8]:
def zero_candidate_is_near_value_axis(candidate, axis_group, orientation, image_width, image_height):
    """
    Decide whether a detected zero is likely to be an axis tick label, not a data label.
    """
    if not is_zero_value(candidate["value"]):
        return False

    if orientation == "vertical":
        # For vertical bar charts, zero should be near the left side and lower half.
        if candidate["cx"] > image_width * 0.45:
            return False
        if candidate["cy"] < image_height * 0.45:
            return False

        if axis_group:
            axis_right = float(np.median([g["right"] for g in axis_group]))
            return abs(candidate["right"] - axis_right) < image_width * 0.18

        return True

    if orientation == "horizontal":
        # For horizontal bar charts, zero is usually left/bottom.
        if candidate["cy"] < image_height * 0.45:
            return False
        if candidate["cx"] > image_width * 0.50:
            return False

        if axis_group:
            axis_y = float(np.median([g["cy"] for g in axis_group]))
            return abs(candidate["cy"] - axis_y) < image_height * 0.15

        return True

    return False


def check_starts_at_zero(image_path, orientation="vertical", return_debug=False):
    """
    Main function.

    Parameters
    ----------
    image_path:
        Path to chart image.
    orientation:
        'vertical' for vertical bar charts.
        'horizontal' for horizontal bar charts.
    return_debug:
        If True, include all OCR detections and axis boxes in the result.
    """
    image = read_image(image_path)
    h, w = image.shape[:2]

    numbers = ocr_numbers_with_tesseract(image, orientation)
    axis_group = find_value_axis(numbers, orientation, w, h)

    axis_type = "y-axis" if orientation == "vertical" else "x-axis"

    if not axis_group:
        result = {
            "status": "compliant",
            "starts_at_zero": True,
            "orientation": orientation,
            "axis_type": axis_type,
            "detected_axis_values": [],
            "zero_source": None,
            "reason": "No reliable value-axis labels detected; treated as compliant by project rule.",
        }

        if return_debug:
            result["all_ocr_numbers"] = numbers
            result["axis_group"] = []

        return result

    axis_values = [item["value"] for item in axis_group]
    axis_values_unique = unique_sorted_values(axis_values)

    zero_in_axis_group = any(is_zero_value(v) for v in axis_values_unique)

    zero_candidates = [
        n for n in numbers
        if zero_candidate_is_near_value_axis(n, axis_group, orientation, w, h)
    ]

    zero_rescue_found = len(zero_candidates) > 0
    contains_zero = zero_in_axis_group or zero_rescue_found

    if zero_in_axis_group:
        zero_source = "axis_group"
        reason = "Zero found directly in the detected value-axis label group."
    elif zero_rescue_found:
        zero_source = "axis_region_rescue"
        reason = "Zero found in the expected zero area near the value axis."
        axis_values_unique = unique_sorted_values(axis_values_unique + [0.0])
    else:
        zero_source = None
        reason = "Value-axis labels were detected, but zero was not found."

    result = {
        "status": "compliant" if contains_zero else "non_compliant",
        "starts_at_zero": bool(contains_zero),
        "orientation": orientation,
        "axis_type": axis_type,
        "detected_axis_values": axis_values_unique,
        "zero_source": zero_source,
        "reason": reason,
    }

    if return_debug:
        result["all_ocr_numbers"] = numbers
        result["axis_group"] = axis_group
        result["zero_candidates"] = zero_candidates

    return result

## 8. Debugging helpers

Use these when the result looks wrong. The most useful one is `draw_ocr_debug(...)`.

In [9]:
def detections_to_dataframe(detections):
    if not detections:
        return pd.DataFrame()

    columns = [
        "value", "text", "confidence",
        "left", "right", "top", "bottom", "cx", "cy",
        "roi", "preprocess", "psm"
    ]

    df = pd.DataFrame(detections)
    existing_columns = [c for c in columns if c in df.columns]
    return df[existing_columns].sort_values(["cy", "cx"]).reset_index(drop=True)


def debug_ocr_table(image_path, orientation="vertical"):
    image = read_image(image_path)
    numbers = ocr_numbers_with_tesseract(image, orientation)
    return detections_to_dataframe(numbers)


def draw_ocr_debug(
    image_path,
    orientation="vertical",
    out_path="debug_tesseract_detections.png",
    show=True
):
    """
    Draw all Tesseract numeric boxes.

    Green boxes = selected value-axis labels.
    Red boxes   = zero candidates near the value axis.
    Blue boxes  = other numeric detections.
    """
    image = read_image(image_path)
    h, w = image.shape[:2]

    result = check_starts_at_zero(
        image_path,
        orientation=orientation,
        return_debug=True
    )

    numbers = result["all_ocr_numbers"]
    axis_group = result["axis_group"]
    zero_candidates = result.get("zero_candidates", [])

    def key(det):
        return (
            round(det["left"], 1),
            round(det["right"], 1),
            round(det["top"], 1),
            round(det["bottom"], 1),
            round(det["value"], 2),
        )

    axis_keys = {key(d) for d in axis_group}
    zero_keys = {key(d) for d in zero_candidates}

    canvas = image.copy()

    for det in numbers:
        k = key(det)

        if k in zero_keys:
            color = (0, 0, 255)      # red
            thickness = 3
        elif k in axis_keys:
            color = (0, 180, 0)      # green
            thickness = 3
        else:
            color = (255, 0, 0)      # blue
            thickness = 1

        x1, y1 = int(det["left"]), int(det["top"])
        x2, y2 = int(det["right"]), int(det["bottom"])

        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, thickness)

        label = f"{det['text']} -> {det['value']:g}"
        cv2.putText(
            canvas,
            label,
            (x1, max(15, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            color,
            1,
            cv2.LINE_AA
        )

    cv2.imwrite(out_path, canvas)

    print("Result:")
    for k, v in result.items():
        if k not in {"all_ocr_numbers", "axis_group", "zero_candidates"}:
            print(f"  {k}: {v}")

    print(f"\nSaved debug image to: {out_path}")

    if show:
        show_image(canvas, title="Tesseract OCR debug")

    return out_path


def save_preprocessing_debug(
    image_path,
    orientation="vertical",
    out_dir="debug_preprocessing"
):
    """
    Save preprocessing variants for every ROI.
    This helps you see whether image quality is the real issue.
    """
    image = read_image(image_path)
    rois = make_axis_rois(image, orientation)

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    saved = []

    for roi in rois:
        x0, y0, x1, y1 = roi["x0"], roi["y0"], roi["x1"], roi["y1"]
        crop = image[y0:y1, x0:x1]

        roi_path = out_dir / f"{roi['name']}_original.png"
        cv2.imwrite(str(roi_path), crop)
        saved.append(str(roi_path))

        for name, processed in preprocess_variants(crop, scale=roi["scale"]):
            path = out_dir / f"{roi['name']}_{name}_scale{roi['scale']}.png"
            cv2.imwrite(str(path), processed)
            saved.append(str(path))

    print("Saved preprocessing debug images:")
    for path in saved:
        print(f"  {path}")

    return saved

## 9. Run on one image

In [18]:
# Change this path to the image you want to test.
IMAGE_PATH = "/Users/nguyenanhvu/Documents/AMD-Semester3/GroupProject/ibcs-ml-base/Dataset/test4.png"

# Use "vertical" for vertical bar charts.
# Use "horizontal" for horizontal bar charts.
ORIENTATION = "vertical"

result = check_starts_at_zero(IMAGE_PATH, orientation=ORIENTATION)
result

{'status': 'compliant',
 'starts_at_zero': True,
 'orientation': 'vertical',
 'axis_type': 'y-axis',
 'detected_axis_values': [-915.0, -20.0, -15.0, -10.0, 0.0, 20.0],
 'zero_source': 'axis_group',
 'reason': 'Zero found directly in the detected value-axis label group.'}

In [ ]:
# Show all OCR numeric detections as a table.
debug_ocr_table(IMAGE_PATH, orientation=ORIENTATION)

In [ ]:
# Draw OCR boxes on the image.
draw_ocr_debug(
    IMAGE_PATH,
    orientation=ORIENTATION,
    out_path="debug_tesseract_detections.png",
    show=True
)

In [ ]:
# Save preprocessing versions of each ROI.
# Use this when Tesseract misses faint/small axis labels.
save_preprocessing_debug(
    IMAGE_PATH,
    orientation=ORIENTATION,
    out_dir="debug_preprocessing"
)

## 10. Optional batch test on a folder

In [ ]:
from glob import glob

def batch_check_starts_at_zero(pattern, orientation="vertical"):
    rows = []

    for image_path in sorted(glob(pattern)):
        try:
            result = check_starts_at_zero(image_path, orientation=orientation)
            rows.append({
                "image_path": image_path,
                "status": result["status"],
                "starts_at_zero": result["starts_at_zero"],
                "detected_axis_values": result["detected_axis_values"],
                "zero_source": result["zero_source"],
                "reason": result["reason"],
            })
        except Exception as exc:
            rows.append({
                "image_path": image_path,
                "status": "error",
                "starts_at_zero": None,
                "detected_axis_values": [],
                "zero_source": None,
                "reason": str(exc),
            })

    return pd.DataFrame(rows)

# Example:
# df = batch_check_starts_at_zero(
#     "/Users/nguyenanhvu/Documents/AMD-Semester3/GroupProject/ibcs-ml-base/Dataset/Compliant/*.png",
#     orientation="vertical"
# )
# df

## Notes

- This version intentionally does **not** use EasyOCR.
- The ROI strategy is stricter than full-image OCR. That is deliberate: full-image OCR often picks up data labels inside bars and confuses them with axis tick labels.
- For percentage charts, `0%` and `0` both become numeric `0.0`.
- If OCR fails on a clear image, run:
  - `draw_ocr_debug(...)`
  - `save_preprocessing_debug(...)`

Those two outputs show whether the problem is OCR quality, cropping, or axis grouping.